# Nemotron-Personas-Korea EDA
**Dataset:** [`nvidia/Nemotron-Personas-Korea`](https://huggingface.co/datasets/nvidia/Nemotron-Personas-Korea) — 1,000,000 rows of LLM-generated synthetic Korean personas (NVIDIA, CC-BY-4.0).

**Project:** 캡스톤디자인 — LLM 멀티 에이전트 기반 가상 도시 마케팅 시뮬레이션

이 노트북은 Google Colab에서 실행하도록 작성되었습니다. 실행 후 `results/` 폴더를 zip으로 다운로드해서
로컬의 `Nemotron-Personas-Korea` 공유 폴더에 넣어주세요.


In [ ]:
# 1. Setup
!pip install -q datasets pyarrow koreanize-matplotlib


In [ ]:
import os
import json
import re
import ast
from collections import Counter
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib  # noqa: F401  (registers a Korean font with matplotlib)

# IMPORTANT: sns.set_theme() resets rcParams (including font.family), so it must be
# called BEFORE koreanize_matplotlib's font is (re)applied — otherwise every Korean
# label renders as tofu boxes (□□□) in the saved PNGs.
sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = koreanize_matplotlib.get_font_name() if hasattr(koreanize_matplotlib, "get_font_name") else plt.rcParams["font.family"]
import matplotlib.font_manager as fm
_korean_fonts = [f.name for f in fm.fontManager.ttflist if any(k in f.name for k in ["Nanum", "Malgun", "AppleGothic", "Noto Sans CJK", "Noto Sans KR"])]
if _korean_fonts:
    plt.rcParams["font.family"] = _korean_fonts[0]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 110

RESULTS_DIR = "results"
FIG_DIR = os.path.join(RESULTS_DIR, "figures")
TABLE_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

summary = {}  # collects headline numbers for summary_stats.json

def save_fig(fig, name):
    path = os.path.join(FIG_DIR, f"{name}.png")
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print("saved:", path)

def save_table(obj, name, index=True):
    path = os.path.join(TABLE_DIR, f"{name}.csv")
    if isinstance(obj, (pd.Series, pd.DataFrame)):
        obj.to_csv(path, index=index, encoding="utf-8-sig")
    else:
        pd.DataFrame(obj).to_csv(path, index=index, encoding="utf-8-sig")
    print("saved:", path)


## 1. 데이터 로드

전체 100만 행을 그대로 써도 되고, Colab 리소스가 부족하면 `SAMPLE_SIZE`를 지정해 일부만 샘플링하세요.
(`None`이면 전체 사용)

In [ ]:
from datasets import load_dataset

SAMPLE_SIZE = None  # 예: 300_000 으로 바꾸면 30만 행만 사용

ds = load_dataset("nvidia/Nemotron-Personas-Korea", split="train")
if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(ds):
    ds = ds.shuffle(seed=42).select(range(SAMPLE_SIZE))

df = ds.to_pandas()
print(df.shape)
df.head(3)


## 2. 데이터 개요 (Overview)

In [ ]:
summary["n_rows"] = int(len(df))
summary["n_cols"] = int(df.shape[1])
summary["columns"] = list(df.columns)

overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "n_missing": df.isna().sum(),
    "pct_missing": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(),
})
save_table(overview, "00_overview", index=True)
overview


In [ ]:
df.sample(20, random_state=42).to_csv(os.path.join(TABLE_DIR, "sample_rows.csv"), index=False, encoding="utf-8-sig")
print("saved sample_rows.csv")


## 3. 인구통계 - 나이 (Age)

In [ ]:
summary["age"] = {
    "mean": float(df["age"].mean()),
    "median": float(df["age"].median()),
    "std": float(df["age"].std()),
    "min": int(df["age"].min()),
    "max": int(df["age"].max()),
}

fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(df["age"], bins=40, kde=True, ax=ax, color="#4C72B0")
ax.set_title(f"연령 분포 (n={len(df):,})")
ax.set_xlabel("나이")
ax.set_ylabel("빈도")
save_fig(fig, "01_age_distribution")


In [ ]:
# 연령대(generation) 파생 변수
df["age_group"] = (df["age"] // 10 * 10).astype(int).astype(str) + "대"
age_group_counts = df["age_group"].value_counts().sort_index()
save_table(age_group_counts, "01_age_group_counts")

fig, ax = plt.subplots(figsize=(8, 5))
age_group_counts.plot(kind="bar", ax=ax, color="#55A868")
ax.set_title("연령대별 인원수")
ax.set_ylabel("인원수")
save_fig(fig, "01_age_group_bar")


## 4. 인구통계 - 범주형 변수 (성별/혼인/병역/주거/학력/전공)

In [ ]:
cat_cols = ["sex", "marital_status", "military_status", "housing_type", "education_level", "bachelors_field"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, col in zip(axes.flat, cat_cols):
    vc = df[col].value_counts()
    save_table(vc, f"02_{col}_counts")
    vc.plot(kind="bar", ax=ax, color="#C44E52")
    ax.set_title(col)
    ax.tick_params(axis="x", rotation=60)
fig.tight_layout()
save_fig(fig, "02_categorical_overview")


## 5. 직업 (Occupation) Top 20

In [ ]:
occ_counts = df["occupation"].value_counts().head(20)
save_table(occ_counts, "03_occupation_top20")

fig, ax = plt.subplots(figsize=(9, 8))
occ_counts.sort_values().plot(kind="barh", ax=ax, color="#8172B2")
ax.set_title("직업 Top 20")
ax.set_xlabel("인원수")
save_fig(fig, "03_occupation_top20")


## 6. 지역 (Province / District) 분포

In [ ]:
province_counts = df["province"].value_counts()
save_table(province_counts, "04_province_counts")

fig, ax = plt.subplots(figsize=(10, 6))
province_counts.plot(kind="bar", ax=ax, color="#64B5CD")
ax.set_title("시/도별 인원수")
ax.set_ylabel("인원수")
ax.tick_params(axis="x", rotation=60)
save_fig(fig, "04_province_bar")


In [ ]:
district_counts = df["district"].value_counts().head(30)
save_table(district_counts, "04_district_top30")

fig, ax = plt.subplots(figsize=(9, 10))
district_counts.sort_values().plot(kind="barh", ax=ax, color="#CCB974")
ax.set_title("시/군/구 Top 30")
ax.set_xlabel("인원수")
save_fig(fig, "04_district_top30")


## 7. 교차분석 (Cross-tabulations)

In [ ]:
# 연령대 x 학력
ct1 = pd.crosstab(df["age_group"], df["education_level"])
ct1 = ct1.loc[sorted(ct1.index, key=lambda x: int(x[:-1]))]
save_table(ct1, "05_agegroup_x_education")

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(ct1, annot=False, cmap="YlGnBu", ax=ax)
ax.set_title("연령대 x 학력 교차표")
save_fig(fig, "05_agegroup_x_education_heatmap")


In [ ]:
# 성별 x 혼인상태
ct2 = pd.crosstab(df["sex"], df["marital_status"], normalize="index") * 100
save_table(ct2.round(2), "05_sex_x_marital_pct")

fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(ct2, annot=True, fmt=".1f", cmap="OrRd", ax=ax)
ax.set_title("성별 x 혼인상태 (행 기준 비율 %)")
save_fig(fig, "05_sex_x_marital_heatmap")


In [ ]:
# 연령대 x 주거형태
ct3 = pd.crosstab(df["age_group"], df["housing_type"])
ct3 = ct3.loc[sorted(ct3.index, key=lambda x: int(x[:-1]))]
save_table(ct3, "05_agegroup_x_housing")

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(ct3, annot=False, cmap="PuBuGn", ax=ax)
ax.set_title("연령대 x 주거형태 교차표")
save_fig(fig, "05_agegroup_x_housing_heatmap")


## 8. 페르소나 텍스트 길이 분석

`professional_persona`, `sports_persona`, `arts_persona`, `travel_persona`, `culinary_persona`,
`family_persona`, `persona`, `cultural_background`, `skills_and_expertise`, `career_goals_and_ambitions`
같은 자유 서술형 텍스트 컬럼들의 길이(글자 수) 분포를 비교합니다.

In [ ]:
text_cols = [
    "professional_persona", "sports_persona", "arts_persona", "travel_persona",
    "culinary_persona", "family_persona", "persona", "cultural_background",
    "skills_and_expertise", "hobbies_and_interests", "career_goals_and_ambitions",
]
text_cols = [c for c in text_cols if c in df.columns]

len_df = pd.DataFrame({c: df[c].astype(str).str.len() for c in text_cols})
len_summary = len_df.describe().T[["mean", "std", "min", "50%", "max"]]
len_summary.columns = ["mean", "std", "min", "median", "max"]
save_table(len_summary.round(1), "06_text_length_summary")

fig, ax = plt.subplots(figsize=(11, 6))
sns.boxplot(data=len_df, ax=ax, orient="h", palette="Set2")
ax.set_title("텍스트 필드별 글자 수 분포")
ax.set_xlabel("글자 수")
save_fig(fig, "06_text_length_boxplot")


## 9. 취미/역량 키워드 빈도 (Hobbies & Skills)

`hobbies_and_interests_list`, `skills_and_expertise_list` 컬럼은 파이썬 리스트를 문자열로 저장한
형태(`"['a', 'b']"`)이므로 `ast.literal_eval`로 파싱한 뒤 빈도를 집계합니다.

In [ ]:
def parse_list_col(series):
    out = []
    for v in series.dropna():
        try:
            items = ast.literal_eval(v) if isinstance(v, str) else v
            if isinstance(items, (list, tuple)):
                out.extend([str(i).strip() for i in items if str(i).strip()])
        except (ValueError, SyntaxError):
            continue
    return out

for col, tag in [("hobbies_and_interests_list", "hobbies"), ("skills_and_expertise_list", "skills")]:
    if col not in df.columns:
        continue
    items = parse_list_col(df[col])
    counts = pd.Series(Counter(items)).sort_values(ascending=False).head(25)
    save_table(counts, f"07_{tag}_top25")

    fig, ax = plt.subplots(figsize=(9, 9))
    counts.sort_values().plot(kind="barh", ax=ax, color="#4C9F70")
    ax.set_title(f"{tag} Top 25")
    ax.set_xlabel("빈도")
    save_fig(fig, f"07_{tag}_top25")


## 10. 범주형 변수 간 연관성 (Cramér's V)

In [ ]:
def cramers_v(x, y):
    confusion = pd.crosstab(x, y)
    chi2 = None
    from scipy.stats import chi2_contingency
    chi2 = chi2_contingency(confusion, correction=False)[0]
    n = confusion.sum().sum()
    r, k = confusion.shape
    phi2 = chi2 / n
    return float(np.sqrt(phi2 / max(min(r - 1, k - 1), 1)))

cat_vars = ["sex", "marital_status", "military_status", "housing_type",
            "education_level", "age_group", "province"]
cat_vars = [c for c in cat_vars if c in df.columns]

mat = pd.DataFrame(np.eye(len(cat_vars)), index=cat_vars, columns=cat_vars)
for a, b in combinations(cat_vars, 2):
    v = cramers_v(df[a], df[b])
    mat.loc[a, b] = v
    mat.loc[b, a] = v
save_table(mat.round(3), "08_cramers_v_matrix")

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(mat, annot=True, fmt=".2f", cmap="rocket_r", vmin=0, vmax=1, ax=ax)
ax.set_title("범주형 변수 간 연관성 (Cramér's V)")
save_fig(fig, "08_cramers_v_heatmap")


## 11. 결과 저장 & 다운로드

`results/` 폴더(figures + tables + summary_stats.json)를 zip으로 압축해서 다운로드한 뒤,
로컬의 `Nemotron-Personas-Korea` 공유 폴더에 풀어 넣어주세요.

In [ ]:
with open(os.path.join(RESULTS_DIR, "summary_stats.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
import shutil
shutil.make_archive("results", "zip", RESULTS_DIR)
print("created results.zip")

try:
    from google.colab import files
    files.download("results.zip")
except ImportError:
    print("Colab이 아니면 좌측 파일 탭에서 results.zip을 직접 다운로드하세요.")
